<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec12_coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec12_coding.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第12回 画像・映像符号化
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec12_coding.m` を Python に移植したもの。第12回のスライド（vie2026-12）で使う図と数値を計算する。データ量とビットレート，ハフマン符号化，線形 PCM（線形再量子化），PSNR，予測符号化（DPCM），動き補償予測（ブロックマッチング），色差サブサンプリング，変換符号化（符号化利得，JPEG の量子化テーブル），ジグザグスキャンを扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

記号は教科書（村松正吾『多次元信号・画像処理の基礎と展開』7.3 節）に合わせる。量子化ステップ $Q$，量子化インデックス（シンボル） $\check{y}$，復号値 $\check{x}$，予測 $\hat{x}$，予測誤差 $d$，動きベクトル $\boldsymbol{m}=(m_\mathrm{v}\ m_\mathrm{h})^\top$。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import fft, linalg
from skimage import io
from skimage.metrics import peak_signal_noise_ratio as psnr   # psnr(参照画像, 評価する画像, data_range=ピーク値)
from skimage.transform import rescale
from skimage.util import img_as_float
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

写真は教科書のサンプル画像 msipimg05（石像の顔，アンコール）をグレースケールにし，$256\times256$ 画素に縮小して使う。空のなめらかな階調に PCM の擬似輪郭や DCT のブロックひずみがはっきり現れ，石像の細かな模様は動き補償のブロックマッチングの手がかりになる。

In [ ]:
X8 = vie.msipimg(5, 256, "gray").astype(np.float64)   # 8 bit（0〜255），256×256
N1, N2 = X8.shape

図の軸ラベルの記号も教科書の組版（$x[\boldsymbol{n}]$ の添え字ベクトルは太字イタリック）に合わせる。

In [ ]:
lbX = r"$x[\boldsymbol{n}]$"
lbY = r"$\check{y}[\boldsymbol{n}]$"
lbQ = r"$Q$"

## データ量とビットレート（教科書の例題「動画像のビットレート」）
画素数 $N_1\times N_2=4320\times 7680$，フレーム間隔 $\Delta_\mathrm{t}=1/60$ s，一配列要素当たり $\beta=8$ bits の RGB カラー動画像。1 フレーム当たりのビット数は $B=3\beta N_1N_2$，ビットレートは $R=B(\Delta_\mathrm{t})^{-1}$ である。

In [ ]:
Nv, Nh, fps, betaBits = 4320, 7680, 60, 8   # fps = (Δt)^{-1}
B = 3 * betaBits * Nv * Nh                  # bits/frame
R = B * fps                                 # bps
print(f"B = {B} bits/frame，R = {R} bps ≃ {R / 1e9:.0f} Gbps")
assert B == 796262400 and R == 47775744000  # 教科書の解答と一致

## JPEG 符号化の圧縮率（教科書の例「JPEG符号化」）
$96\times 96$ 画素，8 bit のグレースケール画像（教科書 図 1.3 (a)）のデータ量と，JPEG で符号化したファイルのデータ量 1066 bytes（教科書の値）から，1 画素当たりのビット数を求める。

In [ ]:
B0 = 96 * 96 * 8 // 8                       # 原画像のデータ量 [bytes]
B1 = 1066                                   # JPEG ファイルのデータ量 [bytes]（教科書の値）
bpp = 8 * B1 / (96 * 96)                    # [bits/pixel]
print(f"B0 = {B0} bytes，{bpp:.2f} bits/pixel")
assert B1 / B0 < 1 / 8                      # 1/8 以下に削減

## ハフマン符号化（教科書の例題「ハフマン符号化」）
シンボル $\check{y}\in\{0,1,2,3\}$ の出現確率 $P[\check{y}]=(0.6,0.25,0.1,0.05)$ から，確率の小さい 2 つを順に併合してハフマンツリーを作り，符号語 $B_{\check{y}}$ を決める（関数 `huffmancode`）。併合のたびに，確率の大きい枝に 0，小さい枝に 1 を割り当てる。

In [ ]:
def huffmancode(p):
    """出現確率 p からハフマン符号の符号語（文字列のリスト）を作る．

    確率の小さい 2 つのノードを併合することを繰り返す．併合のたびに，
    確率の大きい方の枝に 0，小さい方の枝に 1 を符号語の先頭に付け加える．
    """
    code = [""] * len(p)
    nodes = [[k] for k in range(len(p))]       # 各ノードに含まれるシンボル
    prob = list(p)
    while len(prob) > 1:
        idx = sorted(range(len(prob)), key=lambda i: -prob[i])   # 降順（同じ値は元の順）
        prob = [prob[i] for i in idx]
        nodes = [nodes[i] for i in idx]
        hi, lo = nodes[-2], nodes[-1]          # 確率の小さい 2 つ
        for s in hi:
            code[s] = "0" + code[s]
        for s in lo:
            code[s] = "1" + code[s]
        nodes = nodes[:-2] + [hi + lo]
        prob = prob[:-2] + [prob[-2] + prob[-1]]
    return code


p = np.array([0.6, 0.25, 0.1, 0.05])
code = huffmancode(p)
print("符号語:", code)
assert code == ["0", "10", "110", "111"]     # 教科書 図 7.11 (a) の符号語と一致
binrep = [format(k, "02b") for k in range(4)]   # 2 bit 固定長の 2 進数表現
for k in range(len(p)):
    print(f"{k}  {binrep[k]}  {p[k]:.2f}  {code[k]}")

平均符号長 $\ell=\sum_{\check{y}}P[\check{y}]\ell_{\check{y}}$（$\ell_{\check{y}}$：符号語の語長）と，エントロピー $H=-\sum_{\check{y}}P[\check{y}]\log_2P[\check{y}]$（平均符号長の下限）を求める。

In [ ]:
lens = np.array([len(c) for c in code])
Lavg = np.sum(p * lens)                      # 平均符号長
H = -np.sum(p * np.log2(p))                  # エントロピー
print("ℓ =", " + ".join(f"{pk:g}×{lk}" for pk, lk in zip(p, lens)), f"= {Lavg:.2f}")
print(f"H = {H:.2f}")
assert abs(Lavg - 1.55) < 1e-12              # 教科書の解答と一致

シンボル列を 2 進数（2 bit 固定長）とハフマン符号語で表し，ビット数を比べる。ハフマン符号は語頭符号（どの符号語も他の符号語の頭にならない）なので，区切りがなくても一意に復号できる。

In [ ]:
def prefixdecode(bits, code):
    """語頭符号のビット列 bits を先頭から読み，符号語 code に一致したらシンボルを出力する．"""
    seq, buf = [], ""
    for b in bits:
        buf += b
        if buf in code:
            seq.append(code.index(buf))
            buf = ""
    return seq


seq = [0, 3, 1, 0, 0, 0, 0, 0, 1, 2, 1, 0, 0, 0, 1, 1, 0, 0, 0, 2]
sBin = "".join(binrep[s] for s in seq)
sHuff = "".join(code[s] for s in seq)
print("2 進数　:", " ".join(binrep[s] for s in seq))
print("ハフマン:", " ".join(code[s] for s in seq))
nbits = [len(sBin), len(sHuff)]
print("ビット数:", nbits)
assert nbits == [40, 31]                     # 教科書の解答（40 bit と 31 bit）と一致
assert prefixdecode(sHuff, code) == seq      # 区切りなしで復号できる

## 線形再量子化の PSNR（教科書の例題「線形再量子化」）
量子化誤差 $e$ が $[-Q/2,Q/2]$ の一様分布なら，分散は $\sigma_\mathrm{e}^2=Q^2/12$。ピーク値 1 に対する PSNR は $10\log_{10}\sigma_\mathrm{e}^{-2}=10\log_{10}12-20\log_{10}Q$ [dB]。$Q_\alpha=(2^{\alpha}-1)^{-1}$ とすると，1 bit 減らすごとに約 $20\log_{10}2\simeq 6.02$ dB 下がる。

In [ ]:
c12 = 10 * np.log10(12)                      # 10 log10 12 [dB]
d1 = 20 * np.log10(2)                        # 1 bit 当たりの差 [dB]


def psnrTh(a):
    return c12 - 20 * np.log10(1 / (2**a - 1))   # α bit の理論値 [dB]


thPSNR = np.array([psnrTh(4), psnrTh(6)])    # 4 bpp と 6 bpp
dTh = thPSNR[1] - thPSNR[0]                  # = 20 log10(63/15)
print(f"10log10(12) = {c12:.2f} dB，20log10(2) = {d1:.2f} dB")
print(f"理論値：4 bpp {thPSNR[0]:.2f} dB，6 bpp {thPSNR[1]:.2f} dB，差 {dTh:.2f} dB")

一様乱数で仮定を確かめる。$[0,1]$ の一様分布に従う値を四捨五入で再量子化すると，PSNR は理論値にほぼ一致する。教科書 図 7.9 の実画像（34.40 dB，46.79 dB）では，画素値の分布が一様でないため少しずれる。

In [ ]:
rng = np.random.default_rng(0)
u = rng.random(1000000)
al = [4, 6]
simPSNR = np.zeros(2)
for i in range(2):
    Qa = 1 / (2 ** al[i] - 1)
    simPSNR[i] = psnr(u, Qa * np.floor(u / Qa + 0.5), data_range=1)   # 四捨五入で再量子化
print("一様乱数の PSNR:", np.round(simPSNR, 2), "dB")

## 線形 PCM 符号化の処理例（msipimg05）
8 bit の画素値を量子化ステップ $Q$ で $\check{y}[\boldsymbol{n}]=\lfloor x[\boldsymbol{n}]/Q \rfloor$ と量子化し，$\check{x}[\boldsymbol{n}]=Q\,\check{y}[\boldsymbol{n}]+Q/2$ で戻す（区間の中央）。$Q=8$（5 bpp）と $Q=16$（4 bpp）で比べる。

In [ ]:
Qs = [8, 16]
psnrPCM = np.zeros(len(Qs))
Xpcm = []
for i, Q in enumerate(Qs):
    yq = np.floor(X8 / Q)
    Xh = Q * yq + Q / 2
    psnrPCM[i] = psnr(X8, Xh, data_range=255)
    Xpcm.append(Xh)
    print(f"Q = {Q:2d}: PSNR = {psnrPCM[i]:.2f} dB")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], X8.astype(np.uint8), "原画像（8 bpp）")
for ax, Q, Xh, ps in zip(axs[1:], Qs, Xpcm, psnrPCM):
    show(ax, Xh.astype(np.uint8), f"$Q={Q}$（{8 - int(np.log2(Q))} bpp，{ps:.2f} dB）")
plt.show()

1 bit 減らすと PSNR は約 6 dB 下がる（2 桁に丸めた値の差）。

In [ ]:
dPSNR = np.round(psnrPCM[0], 2) - np.round(psnrPCM[1], 2)
print(f"差 {dPSNR:.2f} dB")

## PSNR の数値例（前回スライドの例題）
2 bit（ピーク値 $2^2-1=3$）の $4\times4$ 配列どうしの PSNR。差の二乗のうち零でないものを，行ごとに左から並べて示す。

In [ ]:
Xa = np.array([[0, 1, 1, 0], [1, 2, 3, 1], [1, 3, 2, 1], [0, 1, 1, 0]])
Xb = np.array([[0, 1, 1, 1], [1, 3, 0, 1], [1, 2, 2, 1], [2, 1, 1, 0]])
sq = ((Xa - Xb) ** 2).ravel()                # 行ごとの順に並べる
sq = sq[sq != 0]
mse = np.mean((Xa - Xb) ** 2)
psnrEx = 10 * np.log10(3**2 / mse)
print("差の二乗:", " + ".join(map(str, sq)), f"，N = {Xa.size}")
print(f"MSE = {mse:g}，PSNR = {psnrEx:.2f} dB")
assert list(sq) == [1, 1, 9, 1, 4] and mse == 1 and f"{psnrEx:.2f}" == "9.54"   # スライドの値と一致

## 予測符号化（DPCM）
水平方向の一つ前の復号済み画素から $\hat{x}[n_\mathrm{v},n_\mathrm{h}]=\rho\,\check{x}[n_\mathrm{v},n_\mathrm{h}-1]$（$\rho=0.95$）と予測し，予測誤差 $d=x-\hat{x}$ を $Q=16$ で量子化する（四捨五入）。符号化器の中に局所復号器を持つので，量子化誤差は伝播しない。

各行は独立に処理できるので，全行をまとめて左の列から順に処理する。

In [ ]:
rho, Q = 0.95, 16
Xd = np.zeros((N1, N2))
S = np.zeros((N1, N2))
D = np.zeros((N1, N2))
prev = np.full(N1, 128.0)                    # 各行の先頭は 128 から予測
for c in range(N2):
    xhat = rho * prev                        # 予測
    D[:, c] = X8[:, c] - xhat                # 予測誤差
    S[:, c] = np.round(D[:, c] / Q)          # 量子化
    Xd[:, c] = Q * S[:, c] + xhat            # 局所復号
    prev = Xd[:, c]
Xd = np.clip(Xd, 0, 255)
psnrDPCM = psnr(X8, Xd, data_range=255)
print(f"DPCM: PSNR = {psnrDPCM:.2f} dB")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], X8.astype(np.uint8), "原画像")
show(axs[1], np.round(Xd).astype(np.uint8), f"DPCM $Q=16$（{psnrDPCM:.2f} dB）")
plt.show()

## 予測前後のヒストグラムとエントロピー
原画像の画素値，PCM の量子化インデックス，DPCM の量子化インデックスのエントロピー [bits/pixel] を比べる（関数 `entbits`）。DPCM のインデックスは 0 付近に集中するので，エントロピー符号化の効果が大きい。

In [ ]:
def entbits(v):
    """配列 v の値の出現頻度から求めたエントロピー [bits/要素]．"""
    _, cnt = np.unique(v, return_counts=True)
    pr = cnt / v.size
    return -np.sum(pr * np.log2(pr))


Spcm = np.floor(X8 / 16)
Hs = np.array([entbits(X8), entbits(Spcm), entbits(S)])
print("エントロピー [bits/pixel]：原画像 {:.2f}，PCM {:.2f}，DPCM {:.2f}".format(*Hs))

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(6, 5), layout="constrained")
axs[0].hist(X8.ravel(), np.arange(-0.5, 256), color=cgray)
axs[0].set(xlim=(-1, 256), xlabel=lbX, title=f"原画像（{Hs[0]:.2f} bits/pixel）")
axs[1].hist(Spcm.ravel(), np.arange(-0.5, 16), color=ccool)
axs[1].set(xlim=(-8.5, 16.5), xlabel=lbY, title=f"PCM {lbQ}=16 のインデックス（{Hs[1]:.2f} bits/pixel）")
axs[2].hist(S.ravel(), np.arange(-8.5, 17), color=cmain)
axs[2].set(xlim=(-8.5, 16.5), xlabel=lbY, title=f"DPCM {lbQ}=16 のインデックス（{Hs[2]:.2f} bits/pixel）")
plt.show()

## 動き補償予測の効果（ブロックマッチングによる動き推定）
実際の動画の連続する 2 フレームを使う（前回スライドの映像。スライドの動画 anim-11-motion-a と同じもので，VieWork の data に置いた）。カメラが横に動いているので，内容は水平に動き，手前の木と奥の家では動きの大きさが違う。

前フレームを復号済み参照フレーム $\check{\mathbf{x}}_{m_\mathrm{t}}$ とみなし，現フレーム $\mathbf{x}_{n_\mathrm{t}}$ から探索範囲の余白を除いた $304\times464$ 画素を処理する。

In [ ]:
wY = np.array([0.299, 0.587, 0.114])         # 輝度 Y の重み
Xref = io.imread(vie.datafile("anim-11-motion-a-f0.jpg")).astype(np.float64) @ wY   # 前フレーム（参照フレーム）
Xcur = io.imread(vie.datafile("anim-11-motion-a-f1.jpg")).astype(np.float64) @ wY   # 現フレーム
r0, c0 = 8, 8                                # 処理する範囲の左上（周囲に探索範囲の余白 8 画素）
Hf, Wf = 304, 464                            # 処理する範囲の大きさ（16 の倍数）
F0 = Xref[r0:r0 + Hf, c0:c0 + Wf]            # 前フレームの同じ範囲
F1 = Xcur[r0:r0 + Hf, c0:c0 + Wf]            # 現フレーム

$16\times16$ 画素のブロック $b$ ごとに，探索範囲 $\mathcal{N}_\mathrm{w}=\{-8,\ldots,8\}^2$ の全探索で SAD

$$\mathrm{SAD}(\boldsymbol{m};b)=\sum_{\boldsymbol{n}\in\mathcal{N}_b}|x[\boldsymbol{n}]-\check{x}[\boldsymbol{n}+\boldsymbol{m}]|$$

を最小にする $\hat{\boldsymbol{m}}_b$ を求め，参照フレームの $\boldsymbol{n}+\hat{\boldsymbol{m}}_b$ の画素で予測する（動き補償）。ずらし量 $\boldsymbol{m}$ ごとに全ブロックの SAD をまとめて計算する。

In [ ]:
Bs, w = 16, 8
nbv, nbh = Hf // Bs, Wf // Bs
sadmaps = np.zeros((2 * w + 1, 2 * w + 1, nbv, nbh))   # [m_v + w, m_h + w, ブロック]
for mv in range(-w, w + 1):
    for mh in range(-w, w + 1):
        ref = Xref[r0 + mv:r0 + mv + Hf, c0 + mh:c0 + mh + Wf]
        sadmaps[mv + w, mh + w] = np.abs(F1 - ref).reshape(nbv, Bs, nbh, Bs).sum(axis=(1, 3))
imin = sadmaps.reshape(-1, nbv, nbh).argmin(axis=0)   # SAD が最小になる m の番号
mhat = np.stack(np.unravel_index(imin, (2 * w + 1, 2 * w + 1)), axis=-1) - w   # ブロックごとの動きベクトル (m_v, m_h)
P = np.zeros((Hf, Wf))                       # 動き補償予測
for bi in range(nbv):
    for bj in range(nbh):
        m = mhat[bi, bj]
        ib, jb = bi * Bs, bj * Bs
        P[ib:ib + Bs, jb:jb + Bs] = Xref[r0 + ib + m[0]:r0 + ib + m[0] + Bs, c0 + jb + m[1]:c0 + jb + m[1] + Bs]
vals, cnts = np.unique(mhat[:, :, 1], return_counts=True)   # 水平成分の分布（手前と奥で違う）
print("m_h :", vals)
print("個数:", cnts)

単純なフレーム差分と，動き補償後の予測誤差の分散を比べる。

In [ ]:
dNoMC = F1 - F0                              # フレーム差分
dMC = F1 - P                                 # 動き補償後の予測誤差
varMC = [np.var(dNoMC, ddof=1), np.var(dMC, ddof=1)]
print(f"分散：フレーム差分 {varMC[0]:.0f}，動き補償後 {varMC[1]:.0f}")

模様のはっきりしたブロック（画素値の分散が最大のブロック）を一つ選び，SAD の分布と推定した動きベクトルを図にする。現フレームには選んだブロックを橙の枠で示す。

In [ ]:
bvar = F1.reshape(nbv, Bs, nbh, Bs).transpose(0, 2, 1, 3).reshape(nbv, nbh, -1).var(axis=2, ddof=1)
bsv, bsh = np.unravel_index(np.argmax(bvar), bvar.shape)
msel = mhat[bsv, bsh]
print("選んだブロックの動きベクトル (m_v, m_h) =", tuple(int(v) for v in msel))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2), layout="constrained")
show(axs[0], np.clip(F1, 0, 255).astype(np.uint8), "現フレーム")
axs[0].add_patch(plt.Rectangle((bsh * Bs - 1.5, bsv * Bs - 1.5), Bs + 2, Bs + 2, fill=False, ec=cwarm, lw=2))
show(axs[1], np.clip(np.round(128 + dNoMC / 2), 0, 255).astype(np.uint8), "フレーム差分")
show(axs[2], np.clip(np.round(128 + dMC / 2), 0, 255).astype(np.uint8), "動き補償後の予測誤差")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), layout="constrained")
ax.imshow(sadmaps[:, :, bsv, bsh], cmap="gray_r", extent=(-w - 0.5, w + 0.5, w + 0.5, -w - 0.5))
ax.plot(msel[1], msel[0], "o", ms=7, mew=1.5, mfc="none", color=cwarm)
ax.plot(0, 0, "+", ms=7, mew=1.2, color=ccool)
ax.set(xticks=[-w, 0, w], yticks=[-w, 0, w], xlabel=r"$m_\mathrm{h}$", ylabel=r"$m_\mathrm{v}$")
plt.show()

## 色差サブサンプリング（4:2:0）
msipimg02（花束）の Y, Cb, Cr を 4:2:0 の大きさで並べる。色とりどりの花で色差 Cb, Cr の模様がはっきりしている。上下を切り出した $384\times512$ 画素を $192\times256$ 画素に縮小する。

RGB と YCbCr の変換は ITU-R BT.601 の式（Y は 16〜235，Cb, Cr は 16〜240 の範囲）で書く。縮小・拡大は scikit-image の `rescale`（3 次補間）を使う。

In [ ]:
T601 = np.array([[65.481, 128.553, 24.966], [-37.797, -74.203, 112.0], [112.0, -93.786, -18.214]])
off601 = np.array([16.0, 128.0, 128.0])


def rgb2ycbcr(X):
    """[0,1] の RGB を BT.601 の YCbCr（[0,1] に正規化）に変換する．"""
    return (X @ T601.T + off601) / 255


def ycbcr2rgb(Y):
    """BT.601 の YCbCr（[0,1] に正規化）を RGB に戻す．"""
    return (Y * 255 - off601) @ np.linalg.inv(T601).T


Xc = img_as_float(vie.msipimg(2))
Xc = np.clip(rescale(Xc[64:448], 0.5, order=3, anti_aliasing=True, channel_axis=2), 0, 1)
Ycc = rgb2ycbcr(Xc)
Cb2 = rescale(Ycc[:, :, 1], 0.5, order=3, anti_aliasing=True)   # 縦横 1/2 に間引いた色差
Cr2 = rescale(Ycc[:, :, 2], 0.5, order=3, anti_aliasing=True)
X420 = ycbcr2rgb(np.dstack([Ycc[:, :, 0], rescale(Cb2, 2, order=3), rescale(Cr2, 2, order=3)]))
print("Y:", Ycc[:, :, 0].shape, " Cb:", Cb2.shape, " Cr:", Cr2.shape)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(12, 2.6), layout="constrained", width_ratios=[2, 1, 1, 2])
show(axs[0], Ycc[:, :, 0], "Y")
show(axs[1], Cb2, "Cb")
show(axs[2], Cr2, "Cr")
show(axs[3], np.clip(X420, 0, 1), "4:2:0 から復元")
plt.show()

4:4:4 形式に対するデータ量の比。色差 Cb, Cr の画素数は，間引き率 $M_\mathrm{v}\times M_\mathrm{h}$ で割った数になる。

In [ ]:
Mvh = np.array([[1, 2], [2, 2], [1, 4]])     # 4:2:2，4:2:0，4:1:1 の (M_v, M_h)
ratio = (1 + 2 / Mvh.prod(axis=1)) / 3       # Y と Cb, Cr の合計 / 4:4:4
for fmt, r in zip(["4:2:2", "4:2:0", "4:1:1"], ratio):
    print(f"{fmt}: {Fraction(r).limit_denominator(100)}")

## 符号化利得（教科書の例題「符号化利得」）
相関係数 $\rho=0.95$ の AR(1) 過程に 4 点 DCT を施したときの係数の分散 $\sigma_{\mathrm{y}_p}^2/\sigma_\mathrm{x}^2$ と符号化利得 $G$（相加平均 / 相乗平均）。

In [ ]:
M, rho = 4, 0.95
Rx = linalg.toeplitz(rho ** np.arange(M))
C4 = fft.dct(np.eye(M), norm="ortho", axis=0)   # 4 点 DCT の行列（行が基底ベクトル）
Sy = C4 @ Rx @ C4.T
vy = np.diag(Sy)
G = np.mean(vy) / np.prod(vy) ** (1 / M)
GdB = 10 * np.log10(G)
print("分散:", ", ".join(f"{v:.4f}" for v in vy), f"，10log10 G = {GdB:.2f} dB")
assert np.all(np.abs(vy - [3.7562, 0.1626, 0.0512, 0.0300]) < 5e-5) and abs(GdB - 7.57) < 5e-3   # 教科書の解答と一致

相関がない $\rho=0$ の場合は分散に偏りがなく，$10\log_{10}G=0$ dB になる。

In [ ]:
vy0 = np.diag(C4 @ np.eye(M) @ C4.T)
GdB0 = 10 * np.log10(np.mean(vy0) / np.prod(vy0) ** (1 / M))
print(f"10log10 G = {np.round(GdB0, 2) + 0:g} dB")   # + 0 で -0 を 0 にする

## JPEG の量子化テーブル
輝度用 $Q_\mathrm{L}$ と色差用 $Q_\mathrm{C}$（JPEG 規格の参考値）。左上（低周波）ほど細かく，右下（高周波）ほど粗く量子化する。値の大小を色の濃さで示す。

In [ ]:
QL = np.array([[16, 11, 10, 16, 24, 40, 51, 61], [12, 12, 14, 19, 26, 58, 60, 55],
               [14, 13, 16, 24, 40, 57, 69, 56], [14, 17, 22, 29, 51, 87, 80, 62],
               [18, 22, 37, 56, 68, 109, 103, 77], [24, 35, 55, 64, 81, 104, 113, 92],
               [49, 64, 78, 87, 103, 121, 120, 101], [72, 92, 95, 98, 112, 100, 103, 99]])
QC = 99 * np.ones((8, 8), dtype=int)
QC[:4, :4] = [[17, 18, 24, 47], [18, 21, 26, 66], [24, 26, 56, 99], [47, 66, 99, 99]]
print("Q_L =\n", QL)
print("Q_C =\n", QC)
cmapG = ListedColormap(np.linspace([1, 1, 1], np.array(cmain) * 0.75, 256))   # 白→濃い緑

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.6), layout="constrained")
for ax, Qt, ttl in zip(axs, (QL, QC), (r"輝度成分用 $Q_\mathrm{L}$", r"色差成分用 $Q_\mathrm{C}$")):
    ax.imshow(Qt, cmap=cmapG, vmin=0, vmax=125)
    for r in range(8):
        for c in range(8):
            ax.text(c, r, str(Qt[r, c]), ha="center", va="center", fontsize=8,
                    color="w" if Qt[r, c] > 70 else "k")
    ax.set(xticks=range(8), yticks=range(8), xlabel="水平周波数 →", ylabel="← 垂直周波数", title=ttl)
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
plt.show()

## DCT による変換符号化の処理例
$8\times8$ ブロック DCT の係数を $K Q_\mathrm{L}$ で割って四捨五入し（$K$：スケーリングファクター），逆量子化・逆 DCT で戻す。0 でない係数の割合も調べる。

In [ ]:
def blocks(X):
    """256×256 の配列を 8×8 ブロックの並び（32×32×8×8）にする．"""
    return X.reshape(N1 // 8, 8, N2 // 8, 8).transpose(0, 2, 1, 3)


def unblocks(Bk):
    return Bk.transpose(0, 2, 1, 3).reshape(N1, N2)


Ks = [1, 8]
Xdct = []
for K in Ks:
    Yq = np.round(fft.dctn(blocks(X8 - 128), axes=(2, 3), norm="ortho") / (K * QL))
    Xr = unblocks(fft.idctn(Yq * (K * QL), axes=(2, 3), norm="ortho")) + 128
    Xr = np.clip(Xr, 0, 255)
    ps = psnr(X8, Xr, data_range=255)
    nz = np.mean(Yq != 0)
    Xdct.append((Xr, ps))
    print(f"K={K}: PSNR={ps:.2f} dB, 非零係数 {100 * nz:.1f}%")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], X8.astype(np.uint8), "原画像")
for ax, K, (Xr, ps) in zip(axs[1:], Ks, Xdct):
    show(ax, np.round(Xr).astype(np.uint8), f"$K={K}$（{ps:.2f} dB）")
plt.show()

## ジグザグスキャン
低周波から高周波へ斜めに往復しながら 64 個の係数を並べる順序。

In [ ]:
zz = np.zeros((8, 8), dtype=int)
k = 1
for sdiag in range(15):
    idx = [(r, sdiag - r) for r in range(8) if 0 <= sdiag - r <= 7]
    if sdiag % 2 == 0:
        idx = idx[::-1]                       # 偶数番目の斜めは下から上へ
    for r, c in idx:
        zz[r, c] = k
        k += 1
print(zz)
assert np.array_equal(zz, [[1, 2, 6, 7, 15, 16, 28, 29], [3, 5, 8, 14, 17, 27, 30, 43],
                           [4, 9, 13, 18, 26, 31, 42, 44], [10, 12, 19, 25, 32, 41, 45, 54],
                           [11, 20, 24, 33, 40, 46, 53, 55], [21, 23, 34, 39, 47, 52, 56, 61],
                           [22, 35, 38, 48, 51, 57, 60, 62], [36, 37, 49, 50, 58, 59, 63, 64]])   # スライドの値と一致

## まとめ
- エントロピー符号化（ハフマン符号化）は出現確率の高い値に短い符号を割り当てる可逆な符号化。平均符号長はエントロピー以上
- 線形 PCM は 1 bit 減らすごとに PSNR が約 6 dB 下がる。予測符号化は予測誤差を量子化してエントロピーを下げる
- 動き補償予測は，ブロックマッチングで求めた動きベクトルで参照フレームをずらして予測し，予測誤差を小さくする
- 変換符号化は係数の分散の偏り（符号化利得）を利用し，視覚特性に合わせて係数ごとに量子化する
- JPEG は DCT＋量子化＋ハフマン符号化，MPEG は動き補償予測と DCT のハイブリッド符号化

© Copyright, Shogo MURAMATSU, All rights reserved.